In [27]:
import pandas as pd
import numpy as np

## Map to task employment

In [28]:
# Define the ONET version we will be working with the ONET task statement and occupation / task weights files.
ONET_VERSION = "30_2"

### Anthropic Penetration

In [29]:
def read_tsv(path):
    """
    Load TSV using pandas
    """
    return pd.read_csv(path, delimiter="\t")

# Task statement data from ONET
task_statement_df = read_tsv(
    f"https://www.onetcenter.org/dl_files/database/db_{ONET_VERSION}_text/Task%20Statements.txt"
)

task_statement_df = task_statement_df[["O*NET-SOC Code", "Task ID", "Task"]]

# Task penetration data from Anthropic
anthropic_task_penetration_file_path = "https://huggingface.co/datasets/Anthropic/EconomicIndex/resolve/main/labor_market_impacts/task_penetration.csv"
anthropic_task_penetration_df = pd.read_csv(anthropic_task_penetration_file_path)
anthropic_task_penetration_df.head(2)

anthropic_task_penetration_df = pd.merge(task_statement_df, anthropic_task_penetration_df, left_on="Task", right_on="task", how="inner")
anthropic_task_penetration_df.drop(columns=["Task", "task"], inplace=True)
anthropic_task_penetration_df.drop_duplicates(inplace=True)

### OpenAI Exposure

In [30]:
# Eloundou AI Exposure data from OpenAI
eloundou_task_exposure_file_path = "https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/full_labelset.tsv"
eloundou_task_exposure_df = read_tsv(eloundou_task_exposure_file_path)

eloundou_task_exposure_df = eloundou_task_exposure_df[["O*NET-SOC Code", "Task ID", "beta"]]

# Here E1 and E2 is whether an LLM or LLM powered tool can accelerate the task by 50% or more, Hence we can interpret the beta as the lower bound of AI ability with an acceleration of 50% or more, and we can set the beta for E1 to 0.5, and for E2 to 0.5 as well, since we know that the acceleration is at least 50%, but we don't know how much more it can be.
# E0: beta=0
# E1: beta =1
# E2: beta = 0.5

eloundou_task_exposure_df.loc[np.isclose(eloundou_task_exposure_df["beta"], 1.0), "beta"] = 0.5

In [31]:
task_exposure_df = pd.merge(anthropic_task_penetration_df, eloundou_task_exposure_df, on=["O*NET-SOC Code", "Task ID"], how="inner")
task_exposure_df.rename(columns={"O*NET-SOC Code": "onetsoc_code", "Task ID": "task_id"}, inplace=True)

### Occupation to Task Weights

In [32]:
job_families_df = pd.read_csv("./data/ONET/All_Job_Families.csv")
job_families_df.drop(columns=["Occupation"], inplace=True)
job_families_df.rename(columns={"Code": "onetsoc_code", "Job Family": "job_family"}, inplace=True)

In [33]:
ONET_30_2 = f"https://huggingface.co/datasets/MIT-WAL/job_task_input_share/resolve/main/task_labor_input_mean_estimates/{ONET_VERSION}/ONET_30_2_weight_mode_STANDARD_task_labor_input_mean_estimates.csv"

onet_30_2_df = pd.read_csv(ONET_30_2)

onet_30_2_df = pd.merge(onet_30_2_df, job_families_df, on="onetsoc_code", how="left")
onet_30_2_df

,onetsoc_code,task_id,pi,job_family
0,29-1214.00,22717,0.083990,Healthcare Practitioners and Technical
1,29-1214.00,22723,0.083990,Healthcare Practitioners and Technical
2,29-1214.00,22730,0.083990,Healthcare Practitioners and Technical
3,29-1215.00,7771,0.146511,Healthcare Practitioners and Technical
4,29-1041.00,337,0.221040,Healthcare Practitioners and Technical
...,...,...,...,...
17320,13-1141.00,3360,0.000482,Business and Financial Operations
17321,11-9141.00,7242,0.000175,Management
17322,17-2199.11,16567,0.000227,Architecture and Engineering
17323,13-1141.00,3372,0.000335,Business and Financial Operations


In [34]:
#Combining exposure and task labor input share data
task_exposure_df = pd.merge(task_exposure_df, onet_30_2_df, on = ["onetsoc_code", "task_id"], how="inner")

# Strip the .XX suffix from soc_code
task_exposure_df['onetsoc_code'] = task_exposure_df['onetsoc_code'].str.replace(r'\.\d+$', '', regex=True)
task_exposure_df.rename(columns={"onetsoc_code": "soc_code"}, inplace=True)


In [35]:
task_exposure_df.head(2)

,soc_code,task_id,penetration,beta,pi,job_family
0,11-1011,8823,0.0,0.5,0.047235,Management
1,11-1011,8824,0.0,0.0,0.048477,Management


### OEWS Data

In [36]:
OEWS_state_df = pd.read_csv("./data/OEWS/state_M2024_dl.csv")
OEWS_state_df

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,"23,520","30,660","43,830","64,110","98,810",NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.50,98.03,"51,100","72,870","100,640","142,480","203,900",NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,"104,950","130,950","164,400","221,910",#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,"50,410","74,720","106,330","162,780",#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,"18,270","20,950","26,990","41,760","63,900",True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37604,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7051,Industrial Truck and Tractor Operators,...,15.36,17.00,18.91,"30,710","31,950","31,950","35,360","39,330",NaN,NaN
37605,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7061,Cleaners of Vehicles and Equipment,...,13.12,14.63,18.81,"21,840","24,960","27,290","30,430","39,120",NaN,NaN
37606,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7062,"Laborers and Freight, Stock, and Material Move...",...,15.81,17.66,18.71,"27,970","29,120","32,890","36,740","38,910",NaN,NaN
37607,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7064,"Packers and Packagers, Hand",...,11.98,14.93,17.99,"22,380","22,750","24,920","31,060","37,410",NaN,NaN


---

## State Data

Using all_M_2024 from the OEWS to get the employment per state at the occupation level

Filter on:
- AREA_TYPE: [2,3] 
- O_GROUP: Use `detailed`

In [37]:
# Load
OEWS_all_df = pd.read_csv("./data/OEWS/all_data_M_2024.csv")
OEWS_all_df["AREA_TYPE"] = OEWS_all_df["AREA_TYPE"].astype(int)
OEWS_state_df = OEWS_all_df[(OEWS_all_df["AREA_TYPE"] == 2) | (OEWS_all_df["AREA_TYPE"] == 3)].copy()
OEWS_state_df = OEWS_state_df[OEWS_state_df["O_GROUP"] == "detailed"].copy()

OEWS_state_df["A_MEAN"] = (
    OEWS_state_df["A_MEAN"]
    .astype(str)
    .str.replace(",", "", regex=True)
)
OEWS_state_df["A_MEAN"] = pd.to_numeric(OEWS_state_df["A_MEAN"], errors="coerce")

OEWS_state_df["TOT_EMP"] = (
    OEWS_state_df["TOT_EMP"]
    .astype(str)
    .str.replace(",", "", regex=True)
)
OEWS_state_df["TOT_EMP"] = pd.to_numeric(OEWS_state_df["TOT_EMP"], errors="coerce")

# Filter
OEWS_state_df = OEWS_state_df[
    OEWS_state_df["AREA_TYPE"].isin([2, 3]) &
    (OEWS_state_df["O_GROUP"] == "detailed")
].copy()

# Keep relevant features
OEWS_state_df = OEWS_state_df[
    ["AREA", "OCC_CODE", "TOT_EMP", "A_MEAN"]
].copy()

#AREa is in range 0 - 56, so we can safely convert to int8 to save memory
OEWS_state_df["AREA"] = OEWS_state_df["AREA"].astype(int)
OEWS_state_df["TOT_EMP"] = pd.to_numeric(OEWS_state_df["TOT_EMP"], errors="coerce")

OEWS_state_df.replace({"TOT_EMP": {np.nan: 0}}, inplace=True)
OEWS_state_df.replace({"A_MEAN": {np.nan: 0}}, inplace=True)

OEWS_state_df.rename(columns={
    "AREA": "state_code",
    "OCC_CODE": "soc_code",
    "TOT_EMP": "weight",
    "A_MEAN": "a_mean"
}, inplace=True)

employee_count = OEWS_state_df["weight"].sum()
wage_bill = (OEWS_state_df["weight"] * OEWS_state_df["a_mean"]).sum()

print(f"Total number of employees: {employee_count}")
print(f"Total wage bill: {wage_bill}")



/var/folders/3s/9jsvcvx13b1d8r1zd5h0357m0000gp/T/ipykernel_7946/24643213.py:2: DtypeWarning: Columns (4,13,14,15,16,31) have mixed types. Specify dtype option on import or set low_memory=False.
  OEWS_all_df = pd.read_csv("./data/OEWS/all_data_M_2024.csv")


Total number of employees: 154059520.0
Total wage bill: 10323804474800.0


In [38]:
#Merging exposure with state data
OEWS_state_df = pd.merge(task_exposure_df, OEWS_state_df, on="soc_code", how="inner")
print(OEWS_state_df.shape)

#OEWS_state_df.to_csv("./data/OEWS/processed_state_occ_OEWS.csv", index = False)
OEWS_state_df.head()

# Renormalize pi within each soc_id group
#OEWS_state_df = pd.merge(onet_30_2_df, OEWS_state_df, on="soc_id", how="inner")
OEWS_state_df['pi'] = OEWS_state_df.groupby(['soc_code', "state_code"])['pi'].transform(lambda x: x / x.sum())

#OEWS_state_df.drop(columns=["soc_code"], inplace=True)
OEWS_state_df["weight"] = OEWS_state_df["weight"]*OEWS_state_df["pi"]

employee_count_new = OEWS_state_df["weight"].sum()
wage_bill_new = (OEWS_state_df["weight"] * OEWS_state_df["a_mean"]).sum()
print(f"Total number of employees: {employee_count_new}, percentage captured: {employee_count_new/employee_count:.2%}")
print(f"Total wage bill: {wage_bill_new}, percentage captured: {wage_bill_new/wage_bill:.2%}")

(768323, 9)
Total number of employees: 139222370.00000003, percentage captured: 90.37%
Total wage bill: 9473682683900.002, percentage captured: 91.77%


In [39]:
OEWS_state_df.to_parquet("./data/output/state_map_data.parquet", compression="snappy", index = False)

In [40]:
OEWS_state_df

,soc_code,task_id,penetration,beta,pi,job_family,state_code,weight,a_mean
0,11-1011,8823,0.0,0.5,0.023618,Management,1,19.602658,207190.0
1,11-1011,8823,0.0,0.5,0.023618,Management,2,21.492070,211500.0
2,11-1011,8823,0.0,0.5,0.023618,Management,4,73.214745,213280.0
3,11-1011,8823,0.0,0.5,0.023618,Management,5,53.612088,177460.0
4,11-1011,8823,0.0,0.5,0.023618,Management,6,873.381060,289530.0
...,...,...,...,...,...,...,...,...,...
768318,53-7121,12810,0.0,0.0,0.024870,Transportation and Material Moving,47,3.481772,66440.0
768319,53-7121,12810,0.0,0.0,0.024870,Transportation and Material Moving,48,64.164090,63220.0
768320,53-7121,12810,0.0,0.0,0.024870,Transportation and Material Moving,49,0.994792,49110.0
768321,53-7121,12810,0.0,0.0,0.024870,Transportation and Material Moving,54,4.227866,54380.0


In [41]:
OEWS_state_df.columns

Index(['soc_code', 'task_id', 'penetration', 'beta', 'pi', 'job_family',
       'state_code', 'weight', 'a_mean'],
      dtype='object')

---

## Metro and Non Metro Areas


Using all_M_2024 from the OEWS to get the employment per state at the occupation level

Filter on:
- AREA_TYPE: [4,6] 
- O_GROUP: Use `detailed`

In [42]:
# Load
OEWS_all_df = pd.read_csv("./data/OEWS/all_data_M_2024.csv").copy()
OEWS_all_df["AREA_TYPE"] = OEWS_all_df["AREA_TYPE"].astype(int)
OEWS_MSA_df = OEWS_all_df[(OEWS_all_df["AREA_TYPE"] == 4) | (OEWS_all_df["AREA_TYPE"] == 6)].copy()

OEWS_MSA_df["A_MEAN"] = (
    OEWS_MSA_df["A_MEAN"]
    .astype(str)
    .str.replace(",", "", regex=True)
)
OEWS_MSA_df["A_MEAN"] = pd.to_numeric(OEWS_MSA_df["A_MEAN"], errors="coerce")

OEWS_MSA_df["TOT_EMP"] = (
    OEWS_MSA_df["TOT_EMP"]
    .astype(str)
    .str.replace(",", "", regex=True)
)
OEWS_MSA_df["TOT_EMP"] = pd.to_numeric(OEWS_MSA_df["TOT_EMP"], errors="coerce")

# Filter
OEWS_MSA_df = OEWS_MSA_df[
    OEWS_MSA_df["AREA_TYPE"].isin([4, 6]) &
    (OEWS_MSA_df["O_GROUP"] == "detailed")
].copy()

# Keep relevant features
OEWS_MSA_df = OEWS_MSA_df[
    ["AREA", "OCC_CODE", "TOT_EMP", "A_MEAN", "AREA_TYPE"]
].copy()

#AREa is in range 0 - 56, so we can safely convert to int8 to save memory
OEWS_MSA_df["AREA"] = OEWS_MSA_df["AREA"].astype(int)
OEWS_MSA_df["TOT_EMP"] = pd.to_numeric(OEWS_MSA_df["TOT_EMP"], errors="coerce")

OEWS_MSA_df.replace({"TOT_EMP": {np.nan: 0}}, inplace=True)
OEWS_MSA_df.replace({"A_MEAN": {np.nan: 0}}, inplace=True)


OEWS_MSA_df.rename(columns={
    "AREA": "msa_code",
    "OCC_CODE": "soc_code",
    "TOT_EMP": "weight",
    "A_MEAN": "a_mean",
    "AREA_TYPE": "area_type"
}, inplace=True)

employee_count = OEWS_MSA_df["weight"].sum()
wage_bill = (OEWS_MSA_df["weight"] * OEWS_MSA_df["a_mean"]).sum()

print(f"Total number of employees: {employee_count}")
print(f"Total wage bill: {wage_bill}")



/var/folders/3s/9jsvcvx13b1d8r1zd5h0357m0000gp/T/ipykernel_7946/2003444237.py:2: DtypeWarning: Columns (4,13,14,15,16,31) have mixed types. Specify dtype option on import or set low_memory=False.
  OEWS_all_df = pd.read_csv("./data/OEWS/all_data_M_2024.csv").copy()


Total number of employees: 144965680.0
Total wage bill: 9609300459800.0


In [43]:
OEWS_MSA_df

,msa_code,soc_code,weight,a_mean,area_type
215435,10180,11-1011,50.0,213250.0,4
215436,10180,11-1021,2240.0,99130.0,4
215437,10180,11-2021,140.0,124300.0,4
215438,10180,11-2022,310.0,126370.0,4
215439,10180,11-2032,50.0,92520.0,4
...,...,...,...,...,...
414432,7800001,53-7051,40.0,34220.0,6
414433,7800001,53-7061,50.0,29140.0,6
414434,7800001,53-7062,480.0,33390.0,6
414435,7800001,53-7064,120.0,28470.0,6


In [44]:
#Merging exposure with state data
OEWS_MSA_df = pd.merge(task_exposure_df, OEWS_MSA_df, on="soc_code", how="inner").copy()
print(OEWS_MSA_df.shape)

# Renormalize pi within each soc_id group
OEWS_MSA_df['pi'] = OEWS_MSA_df.groupby(['soc_code', "msa_code"])['pi'].transform(lambda x: x / x.sum())

OEWS_MSA_df["weight"] = OEWS_MSA_df["weight"]*OEWS_MSA_df["pi"]

employee_count_new = OEWS_MSA_df["weight"].sum()
wage_bill_new = (OEWS_MSA_df["weight"] * OEWS_MSA_df["a_mean"]).sum()
print(f"Total number of employees: {employee_count_new}, percentage captured: {employee_count_new/employee_count:.2%}")
print(f"Total wage bill: {wage_bill_new}, percentage captured: {wage_bill_new/wage_bill:.2%}")

(4377735, 10)
Total number of employees: 130962329.99999997, percentage captured: 90.34%
Total wage bill: 8819400296000.01, percentage captured: 91.78%


In [45]:
OEWS_MSA_df.to_parquet("./data/output/msa_map_data.parquet", compression="snappy", index = False)

In [46]:
OEWS_MSA_df.columns

Index(['soc_code', 'task_id', 'penetration', 'beta', 'pi', 'job_family',
       'msa_code', 'weight', 'a_mean', 'area_type'],
      dtype='object')

# Mappings

In [47]:
task_definitions_df = task_statement_df[["Task ID", "Task"]].copy()
task_definitions_df.head(5)

task_definitions_df.to_parquet("./data/output/task_dim.parquet", compression="snappy", index = False)
task_definitions_df.columns

Index(['Task ID', 'Task'], dtype='object')

In [48]:
ONET_30_2 = f"https://huggingface.co/datasets/MIT-WAL/job_task_input_share/resolve/main/task_labor_input_mean_estimates/{ONET_VERSION}/ONET_30_2_weight_mode_STANDARD_task_labor_input_mean_estimates.csv"

onet_30_2_df = pd.read_csv(ONET_30_2)
onet_30_2_df = pd.merge(job_families_df, onet_30_2_df, on = "onetsoc_code", how="inner")
onet_30_2_df['onetsoc_code'] = onet_30_2_df['onetsoc_code'].str.replace(r'\.\d+$', '', regex=True)
onet_30_2_df.head(5)

OEWS_mapping_df = pd.read_csv("./data/OEWS/all_data_M_2024.csv")
OEWS_mapping_df = OEWS_mapping_df[["OCC_CODE", "OCC_TITLE"]]
OEWS_mapping_df.rename(columns={"OCC_CODE": "onetsoc_code", "OCC_TITLE": "onetsoc_title"}, inplace=True)

/var/folders/3s/9jsvcvx13b1d8r1zd5h0357m0000gp/T/ipykernel_7946/3172553664.py:8: DtypeWarning: Columns (4,13,14,15,16,31) have mixed types. Specify dtype option on import or set low_memory=False.
  OEWS_mapping_df = pd.read_csv("./data/OEWS/all_data_M_2024.csv")


In [ ]:
occ_mapping_df = pd.merge(onet_30_2_df, OEWS_mapping_df, on = "onetsoc_code", how="inner")
occ_mapping_df = occ_mapping_df[["onetsoc_code", "onetsoc_title", "job_family"]]

occ_mapping_df.to_parquet("./data/output/occupation_dim.parquet", compression="snappy", index = False)
occ_mapping_df.columns

Index(['onetsoc_code', 'onetsoc_title', 'job_family'], dtype='object')

: 